# 06 · Vida con ahorro y reaseguro cedido

## Vida con ahorro

Es una vida universal tipo B: al morir paga la suma asegurada más el valor de cuenta, y vence a los 65 años. Cuatro decisiones la caracterizan:

1. **El componente de inversión** es el valor de rescate (valor de cuenta menos el cargo por rescate), porque es lo que se paga en todas las circunstancias. Se excluye de ingresos y gastos (par. 85, B123), y sus diferencias contra lo esperado ajustan el CSM (B96(c)).
2. **El compromiso de acreditación** se especifica al inicio: la tasa HQM a un año menos 1,25%, con mínimo de 2,5% (B98–B100). Por eso la tasa futura se proyecta con las tasas forward de la misma curva (B74(b)), y los cambios de tasas, incluida la acreditación real del año, son efectos financieros (B97(a)).
3. **Las unidades de cobertura** suman el cargo esperado por riesgo de muerte y 1% anual del valor de cuenta, que representa el servicio de retorno de inversión (B119B).
4. **Todo el resultado financiero va a resultados** (par. 88(a)).

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 100 else f'{x:,.4f}')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
from ifrs17 import closing as C, disclosures as D, reinsurance
from ifrs17 import assumptions as A
res = C.run()
A.SAVINGS_LIFE.declared_rate

{2023: 0.037, 2024: 0.0395, 2025: 0.034}

In [2]:
filas = []
for o in res['va']['closings']:
    det = o.details.set_index('group')
    for g, r in o.results.items():
        filas.append({'cierre': o.closing, 'grupo': g, 'primas': det.loc[g, 'act_premiums'], 'componente_inversión_pagado': det.loc[g, 'act_investment_component'],
                      'ingreso_por_seguros': r.pnl['insurance_revenue'], 'liberación_CSM': -r.movements['csm_release']['csm'], 'CSM': r.closing.csm,
                      'LRC': r.closing.lrc, 'finanzas': r.pnl['ifie_pnl']})
pd.DataFrame(filas)

,cierre,grupo,primas,componente_inversión_pagado,ingreso_por_seguros,liberación_CSM,CSM,LRC,finanzas
0,2023-12-31,VA-2023-RES,"170,520","2,455","17,203",365,"24,036","82,190","-1,968"
1,2023-12-31,VA-2023-SIG,"4,347,890","197,749","327,853","18,278","2,254,242","1,994,863","-57,011"
2,2024-12-31,VA-2023-RES,"158,830","19,541","30,993",839,"28,748","206,485","7,920"
3,2024-12-31,VA-2023-SIG,"3,952,000","341,601","617,159","37,350","2,486,414","5,421,137","294,456"
4,2024-12-31,VA-2024-RES,"305,280","9,677","36,041",746,"50,081","139,978",375
5,2024-12-31,VA-2024-SIG,"4,167,260","202,760","374,854","17,184","2,175,220","1,945,522","61,490"
6,2025-12-31,VA-2023-RES,"143,960","18,555","29,252","1,292","31,868","319,224","8,699"
7,2025-12-31,VA-2023-SIG,"3,683,320","731,270","547,722","50,608","2,546,278","8,152,535","190,296"
8,2025-12-31,VA-2024-RES,"284,910","30,267","53,501","1,510","56,943","361,957","5,895"
9,2025-12-31,VA-2024-SIG,"3,754,030","426,840","505,163","31,119","2,305,375","4,927,455","26,725"


El ingreso por seguros es una fracción pequeña de las primas, porque casi toda la prima va al valor de cuenta, que es componente de inversión. El CSM se libera despacio, porque quedan décadas de cobertura y de servicio de inversión.

## Reaseguro cedido: cuota parte de 40% sobre vida temporal

Cada tratado anual es un grupo de un solo contrato medido con el GMM (par. 60–70A).

- **CSM:** es el costo neto de comprar la cobertura (par. 65).
- **Componente de recuperación de pérdidas:** cuando un grupo subyacente es oneroso, el tratado reconoce un ingreso igual a la pérdida × 40% y ajusta su CSM (66A, B119D). Lo mismo ocurre cuando el subyacente se vuelve más oneroso (66(c)(i)). Las reversiones del subyacente también se revierten en el tratado.

In [3]:
filas = []
for o in res['re']['closings']:
    for t, r in o['results'].items():
        p = reinsurance.presentation(r)
        filas.append({'cierre': o['closing'], 'tratado': t, 'activo': r.closing.lrc, 'CSM_costo_neto': r.closing.csm,
                      'componente_recuperación': r.closing.loss_component, **p})
pd.DataFrame(filas)

,cierre,tratado,activo,CSM_costo_neto,componente_recuperación,allocation_of_premiums_paid,amounts_recovered,loss_recovery_income,loss_recovery_reversal,net_result,finance_pnl,finance_oci,asset
0,2023-12-31,RE-2023,"818,826","983,949","87,166","-372,190","444,412","90,231","-3,065","72,222","13,219","24,922","818,826"
1,2024-12-31,RE-2023,"1,166,609","806,863","220,719","-739,061","309,041","141,593","-8,040","-430,020","29,955","-41,728","1,166,609"
2,2024-12-31,RE-2024,"762,567","931,347","124,924","-404,078","548,107","127,696","-2,772","144,029","10,564","-21,135","762,567"
3,2025-12-31,RE-2023,"1,275,904","991,003","63,147","-714,899","522,682","-138,248","-19,324","-192,216","47,074","22,348","1,275,904"
4,2025-12-31,RE-2024,"941,568","1,112,064","25,874","-721,308","514,570","-87,941","-11,108","-206,738","27,473","10,291","941,568"
5,2025-12-31,RE-2025,"678,067","1,914,424",0.0000,"-466,644","107,706",0.0000,-0.0000,"-358,938","12,493","-7,650","678,067"


En 2025 el ingreso por recuperación es negativo: los grupos subyacentes revirtieron sus pérdidas y el tratado revierte la recuperación correspondiente (66(c)(ii)). El resultado neto del reaseguro se presenta aparte de los ingresos por seguros y no los reduce (par. 82, 86).